## 大模型返回结构化输出
pydantic举例:
- 默认值:Field(default=0,description="评价分数,默认值为1"),这里default需模型厂商支持,description的默认值描述则是提示词注入
- 枚举:两种
    - Enum:枚举类,需要继承Enum类
    - Literal:枚举字符串,Literal["1","2","3"]
- 可选字段:Optional[int]可为int或None
- 列表:director: list[str] = Field(description="导演")
- 嵌套:actors: list[Actor] = Field(description="演员列表")
- 限制条件:ge=0,le=1,gt=0,lt=1,ne=0,3

In [ ]:
import os
import dotenv
from langchain_deepseek import ChatDeepSeek

dotenv.load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


In [7]:
from pydantic import BaseModel,Field
from enum import Enum

class EvaluationScore(int,Enum):
    """评价分数"""
    Satisfied = 3
    Neutral = 2
    Dissatisfied = 1

class UserEvaluation(BaseModel):
    """用户评价"""
    score: EvaluationScore = Field(description="评价分数")
    #枚举,或者使用Literal["1","2","3"]
    comment: str = Field(description="评价内容")
    confidence: float = Field(description="评价置信度",ge=0,le=1)#限制条件

userComment = "我非常满意,服务质量非常好,推荐推荐"
userComment1 = "玛卡巴卡,好的跟掉粪坑一样"

#with_structured_output方法,根据传入的为TypedDict,pydantic模型,Json Scheme,dataclass返回结构化输出
response = model.with_structured_output(UserEvaluation).invoke(userComment1)

print(response)


score=<EvaluationScore.Satisfied: 1> comment='用户发来一段无意义的、令人困惑的内容，既不构成有效的问题，也不属于任何可评估的任务。这种情况下无法提供有价值的回应。' confidence=0.9


In [2]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    """演员"""
    name: str = Field(description="演员姓名")
    age: int = Field(description="演员年龄")
    sex: str = Field(description="演员性别")

class Movie(BaseModel):
    """电影"""
    title: str = Field(description="电影标题")
    director: list[str] = Field(description="导演")
    actors: list[Actor] = Field(description="演员列表")
    release: str = Field(description="上映日期")
    budget: float = Field(description="预算")
    revenue: float = Field(description="票房")
    rating: float = Field(description="评分")

model.with_structured_output(Movie).invoke("介绍一下肖申克的救赎")

Movie(title='肖申克的救赎', director=['弗兰克·德拉邦特'], actors=[Actor(name='蒂姆·罗宾斯', age=66, sex='男'), Actor(name='摩根·弗里曼', age=87, sex='男')], release='1994-09-10', budget=25000000.0, revenue=73300000.0, rating=9.7)